# Business Performance Analytics Platform

This project compares six technology companies based on operating performance and is designed to support management review. It serves as a learning and practice tool for strengthening SQL querying skills, using Python and AI to extract and prepare the necessary data, and applying financial metrics to evaluate business performance. The project also focuses on using data-driven analysis to identify trends, areas of management attention, and potential business recommendations.

The SQL below calculates the numbers. Interpretation cells are left blank so the read can be written from the tables.

**Business question**

How is each company performing across growth, operations, profitability, cash generation, and financial flexibility, and where should management attention be focused?

**Companies**

| Company | Sector |
|---|---|
| Microsoft | Software & Cloud |
| Apple | Consumer Technology |
| NVIDIA | Semiconductors |
| Adobe | Software |
| AMD | Semiconductors |
| CrowdStrike | Cybersecurity |

Each company's own fiscal year is used. Years are not forced onto one calendar.

**Framework**

1. Growth & Momentum
2. Profitability
3. Operating Efficiency
4. Cash Flow & Cash Conversion
5. Financial Health
6. Peer Benchmarking
7. Managment Attention


**Pipeline**

SEC 10-K filings
Python prep 
financial_analytics.clean.company_financials 
SQL views 
interpretation 

Views used: growth_trend, performance, financial_health, peer_benchmarking, company_analysis, and executive_summary for the app.


## Data Preparation & Quality

### Business Question

Is the data complete enough to support the operating metrics?

Source table: `financial_analytics.clean.company_financials`

Missing values are left null. They are not filled with zero.

AMD Total Liabilities was missing in the source file. Where needed it is calculated as:

Total Liabilities = Total Assets − Stockholders’ Equity


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS financial_analytics.data_quality;

CREATE OR REPLACE VIEW financial_analytics.data_quality.sec_company_financials_null_summary AS

SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN company IS NULL THEN 1 ELSE 0 END) AS company_nulls,
    SUM(CASE WHEN ticker IS NULL THEN 1 ELSE 0 END) AS ticker_nulls,
    SUM(CASE WHEN statement IS NULL THEN 1 ELSE 0 END) AS statement_nulls,
    SUM(CASE WHEN fiscal_year IS NULL THEN 1 ELSE 0 END) AS fiscal_year_nulls,
    SUM(CASE WHEN Metric IS NULL THEN 1 ELSE 0 END) AS Metric_nulls,
    SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) AS value_nulls,
    ROUND(SUM(CASE WHEN company IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS company_null_pct,
    ROUND(SUM(CASE WHEN ticker IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS ticker_null_pct,
    ROUND(SUM(CASE WHEN statement IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS statement_null_pct,
    ROUND(SUM(CASE WHEN fiscal_year IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS fiscal_year_null_pct,
    ROUND(SUM(CASE WHEN Metric IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Metric_null_pct,
    ROUND(SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS value_null_pct
FROM financial_analytics.clean.sec_company_financials_clean;


In [0]:
%sql
SELECT * FROM financial_analytics.data_quality.sec_company_financials_null_summary;


total_records,company_nulls,ticker_nulls,statement_nulls,fiscal_year_nulls,Metric_nulls,value_nulls,company_null_pct,ticker_null_pct,statement_null_pct,fiscal_year_null_pct,Metric_null_pct,value_null_pct
1657,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00


In [0]:
%sql
Update financial_analytics.clean.company_financials

set Total_Liabilities = Total_Assets - Total_Stockholders_Equity
where Company = 'AMD'
and Total_Liabilities is null;


num_affected_rows
5


### Company-year quality checks

Each row is one control. Counts come from the clean table.


In [0]:
%sql
CREATE OR REPLACE VIEW financial_analytics.data_quality.company_year_quality AS
SELECT
    COUNT(*) AS total_company_years,
    SUM(CASE WHEN Revenue IS NULL THEN 1 ELSE 0 END) AS missing_revenue,
    SUM(CASE WHEN Operating_Income IS NULL THEN 1 ELSE 0 END) AS missing_operating_income,
    SUM(CASE WHEN Net_Income IS NULL THEN 1 ELSE 0 END) AS missing_net_income,
    SUM(CASE WHEN Operating_Expenses IS NULL THEN 1 ELSE 0 END) AS missing_operating_expenses,
    SUM(CASE WHEN Cash_Flow_from_Operating_Activities IS NULL THEN 1 ELSE 0 END) AS missing_operating_cash_flow,
    SUM(CASE WHEN Total_Current_Assets IS NULL THEN 1 ELSE 0 END) AS missing_current_assets,
    SUM(CASE WHEN Total_Current_Liabilities IS NULL THEN 1 ELSE 0 END) AS missing_current_liabilities,
    SUM(CASE WHEN Total_Assets IS NULL THEN 1 ELSE 0 END) AS missing_total_assets,
    SUM(CASE WHEN Total_Liabilities IS NULL THEN 1 ELSE 0 END) AS missing_total_liabilities,
    SUM(CASE WHEN Total_Stockholders_Equity IS NULL THEN 1 ELSE 0 END) AS missing_equity,
    SUM(CASE WHEN Total_Stockholders_Equity <= 0 THEN 1 ELSE 0 END) AS zero_or_negative_equity,
    SUM(CASE WHEN Short_Term_Debt IS NULL AND Long_Term_Debt IS NULL THEN 1 ELSE 0 END) AS missing_debt,
    SUM(CASE WHEN Total_Current_Liabilities = 0 THEN 1 ELSE 0 END) AS zero_current_liabilities
FROM financial_analytics.clean.company_financials;


In [0]:
%sql
SELECT *
FROM financial_analytics.data_quality.company_year_quality;


total_company_years,missing_revenue,missing_operating_income,missing_net_income,missing_operating_expenses,missing_operating_cash_flow,missing_current_assets,missing_current_liabilities,missing_total_assets,missing_total_liabilities,missing_equity,zero_or_negative_equity,missing_debt,zero_current_liabilities
33,0,0,0,4,0,0,0,0,0,0,0,0,0


## SQL views




In [0]:
%sql
CREATE OR REPLACE VIEW financial_analytics.analytics.growth_trend AS 
WITH growth AS ( 
SELECT 
Company,  
Fiscal_Year, 
Revenue, 
Net_Income, 
(Revenue - LAG(Revenue) OVER (PARTITION By Company ORDER BY Fiscal_Year))
/ NULLIF(LAG(Revenue) OVER (PARTITION BY Company ORDER BY Fiscal_Year), 0) AS Growth_Rate
FROM financial_analytics.clean.company_financials
),

growth_metrics AS (
SELECT Company,  
Fiscal_Year, 
Revenue, 
Net_Income, 
growth_rate,
LAG(growth_rate) OVER (PARTITION BY Company ORDER BY Fiscal_Year) AS Prior_Growth_Rate,
growth_rate - LAG(growth_rate) OVER (PARTITION BY Company ORDER BY Fiscal_Year) AS Growth_Differential
FROM growth
)

SELECT 
Company,  
Fiscal_Year, 
Revenue, 
Net_Income, 
round(Growth_Rate,3) AS Growth_Rate, 
round(Prior_Growth_Rate,3) AS Prior_Growth_Rate,
round(Growth_Differential,3) AS Growth_Differential, 
round(Net_Income / NULLIF(Revenue, 0),3) AS Net_Income_Margin_pct
FROM growth_metrics


In [0]:
%sql
CREATE OR REPLACE VIEW financial_analytics.analytics.performance AS
WITH base AS (
SELECT
c.Company,
c.Fiscal_Year,
c.Revenue,
c.Gross_Profit,
c.Net_Income,
c.Operating_Income,
c.Operating_Expenses,
c.Cash_Flow_from_Operating_Activities,
c.Capital_Expenditures,
LAG(c.Operating_Income) OVER (PARTITION BY c.Company ORDER BY c.Fiscal_Year) AS Prior_Operating_Income,
LAG(c.Net_Income) OVER (PARTITION BY c.Company ORDER BY c.Fiscal_Year) AS Prior_Net_Income,
LAG(c.Operating_Expenses) OVER (PARTITION BY c.Company ORDER BY c.Fiscal_Year) AS Prior_Opex,
LAG(c.Cash_Flow_from_Operating_Activities) OVER (PARTITION BY c.Company ORDER BY c.Fiscal_Year) AS Prior_OCF
FROM financial_analytics.clean.company_financials c
)

SELECT
b.Company,
b.Fiscal_Year,
b.Revenue,
b.Gross_Profit,
b.Net_Income,
b.Operating_Income, 
gt.Net_Income_Margin_pct, 
b.Operating_Expenses,
b.Cash_Flow_from_Operating_Activities,
b.Capital_Expenditures,
round(b.Gross_Profit / NULLIF(b.Revenue, 0),3) AS Gross_Margin_pct,
round(b.Operating_Income / NULLIF(b.Revenue, 0),3) AS Operating_Margin_pct,  
round((b.Operating_Income - b.Prior_Operating_Income) / NULLIF(b.Prior_Operating_Income, 0),3) AS Operating_Income_Growth,
round((b.Net_Income - b.Prior_Net_Income) / NULLIF(b.Prior_Net_Income, 0),3) AS Net_Income_Growth,
round((b.Operating_Expenses - b.Prior_Opex) / NULLIF(b.Prior_Opex, 0),3) AS Operating_Expense_Growth,
round(b.Operating_Expenses / NULLIF(b.Revenue, 0),3) AS Operating_Expense_to_Revenue,
round(gt.Growth_Rate - ((b.Operating_Expenses - b.Prior_Opex) / NULLIF(b.Prior_Opex, 0)),3) AS Revenue_Growth_Minus_Opex_Growth,
round(b.Cash_Flow_from_Operating_Activities / NULLIF(b.Revenue, 0),3) AS Operating_Cash_Flow_Margin_pct,
round((b.Cash_Flow_from_Operating_Activities - b.Prior_OCF) / NULLIF(b.Prior_OCF, 0),3) AS Operating_Cash_Flow_Growth,
round(b.Cash_Flow_from_Operating_Activities / NULLIF(b.Net_Income, 0),3) AS Cash_Conversion_Ratio,
round(b.Cash_Flow_from_Operating_Activities - b.Capital_Expenditures,3) AS Free_Cash_Flow,
round((b.Cash_Flow_from_Operating_Activities - b.Capital_Expenditures) / NULLIF(b.Revenue, 0),3) AS Free_Cash_Flow_Margin_pct
FROM base b
LEFT JOIN financial_analytics.analytics.growth_trend gt
  ON b.Company = gt.Company
  AND b.Fiscal_Year = gt.Fiscal_Year


In [0]:
%sql
CREATE OR REPLACE VIEW financial_analytics.analytics.financial_health AS 
SELECT 
Company,
Fiscal_Year,
Total_Assets,
Total_Liabilities,
Total_Stockholders_Equity,
Total_Current_Assets,
Total_Current_Liabilities,
round(Total_Current_Assets / NULLIF(Total_Current_Liabilities, 0),3) AS Current_ratio,
COALESCE(Short_Term_Debt, 0) + COALESCE(Long_Term_Debt, 0) AS Total_Debt,
round((COALESCE(Short_Term_Debt, 0) + COALESCE(Long_Term_Debt, 0)) / NULLIF(Total_Stockholders_Equity, 0),3) AS debt_to_equity_ratio,
round(Total_Liabilities / NULLIF(Total_Assets, 0),3) AS Liabilities_to_Assets,
round(Net_Income / NULLIF(Total_Stockholders_Equity, 0),3) AS ROE,
round(Net_Income / NULLIF(Total_Assets, 0),3) AS ROA,
round(Net_Income / NULLIF((Total_Assets + LAG(Total_Assets) OVER (PARTITION BY Ticker ORDER BY Fiscal_Year)) / 2, 0),3) AS ROA_Avg,
round(Net_Income / NULLIF((Total_Stockholders_Equity + LAG(Total_Stockholders_Equity) OVER (PARTITION BY Ticker ORDER BY Fiscal_Year)) / 2, 0),3) AS ROE_Avg,
Cash_Flow_from_Operating_Activities

FROM financial_analytics.clean.company_financials


In [0]:
%sql
CREATE OR REPLACE VIEW financial_analytics.analytics.peer_benchmarking AS

WITH ranked_metrics AS (
    SELECT
        gt.company,
        p.fiscal_year,
        RANK() OVER (
            PARTITION BY p.fiscal_year
            ORDER BY gt.Growth_Rate DESC
        ) AS Growth_Rate_Rank,

        RANK() OVER (
            PARTITION BY p.fiscal_year
            ORDER BY gt.Growth_Differential DESC
        ) AS Growth_Differential_Rank,

        RANK() OVER (
            PARTITION BY p.fiscal_year
            ORDER BY p.operating_margin_pct DESC
        ) AS Operating_Margin_Rank,

        RANK() OVER (
            PARTITION BY p.fiscal_year
            ORDER BY p.net_income_margin_pct DESC
        ) AS Net_Income_Margin_Rank,

        RANK() OVER (
            PARTITION BY p.fiscal_year
            ORDER BY p.operating_cash_flow_margin_pct DESC
        ) AS Operating_Cash_Flow_Margin_Rank,

        RANK() OVER (
            PARTITION BY p.fiscal_year
            ORDER BY p.Operating_Expense_Growth ASC
        ) AS Operating_Expense_Growth_Rank,

        RANK() OVER (
            PARTITION BY p.fiscal_year
            ORDER BY fh.current_ratio DESC
        ) AS Current_Ratio_Rank,

        RANK() OVER (
            PARTITION BY p.fiscal_year
            ORDER BY fh.debt_to_equity_ratio ASC
        ) AS Debt_to_Equity_Rank,

        RANK() OVER (
            PARTITION BY p.fiscal_year
            ORDER BY fh.Liabilities_to_Assets ASC
        ) AS Liabilities_to_Assets_Rank,

        RANK() OVER (
            PARTITION BY p.fiscal_year
            ORDER BY fh.roa DESC
        ) AS ROA_Rank,

        RANK() OVER (
            PARTITION BY p.fiscal_year
            ORDER BY fh.roe DESC
        ) AS ROE_Rank

    FROM financial_analytics.analytics.performance p

    LEFT JOIN financial_analytics.analytics.financial_health fh
        ON p.company = fh.company
        AND p.fiscal_year = fh.fiscal_year

    LEFT JOIN financial_analytics.analytics.growth_trend gt
        ON p.company = gt.company
        AND p.fiscal_year = gt.fiscal_year

    WHERE gt.fiscal_year = 2025
)

SELECT
    company,
    fiscal_year,
    Growth_Rate_Rank,
    Growth_Differential_Rank,
    Operating_Margin_Rank,
    Net_Income_Margin_Rank,
    Operating_Cash_Flow_Margin_Rank,
    Operating_Expense_Growth_Rank,
    Current_Ratio_Rank,
    Debt_to_Equity_Rank,
    Liabilities_to_Assets_Rank,
    ROA_Rank,
    ROE_Rank,
    ROUND((
        Growth_Rate_Rank +
        Growth_Differential_Rank +
        Operating_Margin_Rank +
        Net_Income_Margin_Rank +
        Operating_Cash_Flow_Margin_Rank +
        Current_Ratio_Rank +
        Debt_to_Equity_Rank +
        ROA_Rank +
        ROE_Rank
    ) / 9.0, 3) AS Average_Rank
FROM ranked_metrics;


In [0]:
%sql

CREATE OR REPLACE VIEW financial_analytics.analytics.executive_summary AS
WITH latest AS (
SELECT Company, MAX(Fiscal_Year) AS Latest_Fiscal_Year
FROM financial_analytics.analytics.performance
GROUP BY Company
)

SELECT
    p.Company,
    p.Fiscal_Year,
    gt.Revenue,
    gt.Growth_Rate,
    gt.Prior_Growth_Rate,
    gt.Growth_Differential,
    p.Gross_Margin_pct,
    p.Operating_Margin_pct,
    p.Net_Income_Margin_pct,
    p.Operating_Income_Growth,
    p.Net_Income_Growth,
    p.Operating_Expense_Growth,
    p.Operating_Expense_to_Revenue,
    p.Revenue_Growth_Minus_Opex_Growth,
    p.Operating_Cash_Flow_Margin_pct,
    p.Operating_Cash_Flow_Growth,
    p.Cash_Conversion_Ratio,
    p.Free_Cash_Flow,
    p.Free_Cash_Flow_Margin_pct,
    fh.Current_ratio,
    fh.debt_to_equity_ratio,
    fh.Liabilities_to_Assets,
    fh.ROA,
    fh.ROE,
    fh.ROA_Avg,
    fh.ROE_Avg,
    pb.Growth_Rate_Rank,
    pb.Operating_Margin_Rank,
    pb.Net_Income_Margin_Rank,
    pb.Operating_Expense_Growth_Rank,
    pb.Operating_Cash_Flow_Margin_Rank,
    pb.ROA_Rank,
    pb.ROE_Rank
FROM financial_analytics.analytics.performance p
INNER JOIN latest
    ON p.Company = latest.Company
    AND p.Fiscal_Year = latest.latest_fiscal_year 
LEFT JOIN financial_analytics.analytics.growth_trend gt
    ON p.Company = gt.Company
    AND p.Fiscal_Year = gt.Fiscal_Year
LEFT JOIN financial_analytics.analytics.financial_health fh
    ON p.Company = fh.Company
    AND p.Fiscal_Year = fh.Fiscal_Year
LEFT JOIN financial_analytics.analytics.peer_benchmarking pb
    ON p.Company = pb.Company
    AND p.Fiscal_Year = pb.Fiscal_Year;



##  Growth & Momentum

### Business Question

Is the company growing, and is that growth accelerating or slowing?

### Approach

View: `growth_trend`

- Growth_Rate = (Current Revenue − Prior Revenue) / Prior Revenue
- Prior_Growth_Rate = previous year's growth rate
- Growth_Differential = current growth rate − prior growth rate


In [0]:
%sql
SELECT Company, fiscal_year, Revenue, Growth_Rate, Prior_Growth_Rate, Growth_Differential
FROM financial_analytics.analytics.growth_trend 
ORDER BY Company, Fiscal_Year;


Company,fiscal_year,Revenue,Growth_Rate,Prior_Growth_Rate,Growth_Differential
AMD,2021,1.6434E10,null,null,null
AMD,2022,2.3601E10,0.436,null,null
AMD,2023,2.268E10,-0.039,0.436,-0.475
AMD,2024,2.5785E10,0.137,-0.039,0.176
AMD,2025,3.4639E10,0.343,0.137,0.206
Adobe,2021,1.5785E10,null,null,null
Adobe,2022,1.7606E10,0.115,null,null
Adobe,2023,1.9409E10,0.102,0.115,-0.013
Adobe,2024,2.1505E10,0.108,0.102,0.006
Adobe,2025,2.3769E10,0.105,0.108,-0.003


In [0]:
%sql
SELECT 
Company,
fiscal_year,
Growth_Differential,
Growth_Rate
FROM financial_analytics.analytics.growth_trend
WHERE fiscal_year >=2024
ORDER BY company


Company,fiscal_year,Growth_Differential,Growth_Rate
AMD,2024,0.176,0.137
AMD,2025,0.206,0.343
Adobe,2024,0.006,0.108
Adobe,2025,-0.003,0.105
Apple,2024,0.048,0.02
Apple,2025,0.044,0.064
CrowdStrike,2024,-0.181,0.363
CrowdStrike,2025,-0.069,0.294
CrowdStrike,2026,-0.077,0.217
Microsoft,2024,0.088,0.157


## Interpretation
AMD continues to improve its revenue performance, with revenue growth increasing from 13.7% to 34.3%. Its growth differential increased by approximately 21 percentage points, indicating that revenue growth has accelerated rather than slowed.

Adobe has maintained moderate and stable revenue growth, with growth declining slightly from 10.8% to 10.5%. Its growth differential remained relatively consistent, indicating limited change in its revenue growth trajectory.
A
pple’s revenue growth has remained relatively consistent with limited fluctuation. The company has also maintained slightly positive growth momentum, indicating a stable revenue performance profile with gradual improvement in its growth trajectory.

CrowdStrike has had a moderate growth rate from the prior few year however they counitue to lose momentem which could be a problem in the future if the company keeps this doward trend 

Microsoft's growth rate shows a consistent and stable revenue performance, with relatively little fluctuation in growth acceleration. This suggests that Microsoft has maintained steady revenue growth rather than experiencing significant changes in its growth trajectory.

NVIDIA continues to demonstrate strong top-line growth year over year. However, its growth differential declined by approximately 49 percentage points as revenue growth moderated from 114% to 65%. This reflects a significant normalization from its exceptionally high prior-year growth rate rather than a decline in overall revenue.

## Profitability

### Business Question

Is growth turning into profitable operations?

### Approach

View: `performance`

- Gross_Margin_pct = Gross Profit / Revenue
- Operating_Margin_pct = Operating Income / Revenue
- Net_Income_Margin_pct = Net Income / Revenue


In [0]:
%sql
SELECT Company, Fiscal_Year,Gross_Margin_pct ,Net_Income_Margin_pct ,Operating_Margin_pct,Operating_Income_Growth
FROM financial_analytics.analytics.performance
Where Fiscal_Year >= 2024
ORDER BY company, Fiscal_Year;


Company,Fiscal_Year,Gross_Margin_pct,Net_Income_Margin_pct,Operating_Margin_pct,Operating_Income_Growth
AMD,2024,0.494,0.064,0.074,3.738
AMD,2025,0.495,0.125,0.107,0.944
Adobe,2024,0.89,0.259,0.313,0.014
Adobe,2025,0.893,0.3,0.366,0.291
Apple,2024,0.462,0.24,0.315,0.078
Apple,2025,0.469,0.269,0.32,0.08
CrowdStrike,2024,0.752,0.024,-0.006,-0.899
CrowdStrike,2025,0.75,-0.004,-0.029,5.081
CrowdStrike,2026,0.747,-0.034,-0.061,1.52
Microsoft,2024,0.698,0.36,0.446,0.236


## Interpretation

AMD has maintained relatively stable gross margins while operating income and net income have grown year over year. However, operating income growth has declined significantly from 307% in the prior year, indicating that profitability is still improving but at a slower pace.

Adobe has maintained a stable yet strong gross margin (89%) while operating and net income have grown year over year. Operating margin suggests that revenue growth is translating into improving profitability.

Apple has maintained a stable gross margin while operating income has remained relatively consistent, while net income has grown 45%. This suggests that factors beyond operating income growth are contributing to the increase in net income.

CrowdStrike has maintained a relatively consistent gross margin over the past several years; however, operating performance has continued to decline, with operating margins falling from -0.6% to -2.9% and then -6.1%. This indicates increasing operating pressure that is negatively affecting profitability.

Microsoft’s gross margin has gradually declined while operating income has continued to grow year over year. This suggests that Microsoft is generating stronger operating income despite some pressure on gross profitability.

NVIDIA’s gross margin has fluctuated gradually but remains relatively consistent, while operating margin increased sharply from 54% to 62% before declining to around 60%. Net income growth has also moderated, suggesting that NVIDIA’s exceptional profitability and revenue growth are beginning to normalize after its significant prior-year increase.

##Operating Efficiency

### Business Question

Is the business scaling efficiently, or are operating costs growing faster than revenue?

### Approach

View: `performance`

- Operating_Expense_Growth vs prior year
- Operating_Expense_to_Revenue
- Revenue_Growth_Minus_Opex_Growth = Growth_Rate − Operating_Expense_Growth

If revenue growth is higher than opex growth, efficiency is improving. If opex growth is higher, that is cost pressure to investigate.


In [0]:
%sql
SELECT p.Company, p.Fiscal_Year, p.Revenue,gt.Growth_Rate,p.Operating_Expense_Growth, p.Operating_Expense_to_Revenue, p.Revenue_Growth_Minus_Opex_Growth
FROM financial_analytics.analytics.performance p
join financial_analytics.analytics.growth_trend gt
on p.company=gt.company and p.fiscal_year=gt.fiscal_year
where p.Fiscal_Year >= 2024
ORDER BY Company, Fiscal_Year;


Company,Fiscal_Year,Revenue,Growth_Rate,Operating_Expense_Growth,Operating_Expense_to_Revenue,Revenue_Growth_Minus_Opex_Growth
AMD,2024,2.5785E10,0.137,0.076,0.42,0.061
AMD,2025,3.4639E10,0.343,0.243,0.389,0.1
Adobe,2024,2.1505E10,0.108,0.192,0.577,-0.084
Adobe,2025,2.3769E10,0.105,0.009,0.526,0.096
Apple,2024,3.91035E11,0.02,0.048,0.147,-0.028
Apple,2025,4.16161E11,0.064,0.082,0.149,-0.018
CrowdStrike,2024,3.055555E9,0.363,0.265,0.758,0.098
CrowdStrike,2025,3.953624E9,0.294,0.33,0.779,-0.036
CrowdStrike,2026,4.812005E9,0.217,0.262,0.808,-0.045
Microsoft,2024,2.45122E11,0.157,0.07,0.251,0.087


## Interpretation

AMD’s operating efficiency improved as revenue grew faster than operating expenses. Operating expenses also fell from 42.0% to 38.9% of revenue, showing that the company is managing its costs more effectively.

Adobe’s operating efficiency improved significantly in 2025 as operating expense growth slowed to 0.9% while revenue continued to grow. Operating expenses also fell from 57.7% to 52.6% of revenue, showing better cost management.

Apple’s operating expenses have grown faster than revenue in both years, creating some pressure on costs. In 2025, operating expenses grew 1.8 percentage points faster than revenue, while operating expenses remained relatively low at about 15% of revenue.

CrowdStrike’s operating efficiency has weakened as operating expenses have grown faster than revenue. Operating expenses reached 80.8% of revenue in 2026, showing increasing pressure from operating costs.

Microsoft has maintained strong operating efficiency, with revenue growing faster than operating expenses. Operating expenses also fell from 25.1% to 21.2% of revenue, showing that the company is managing its costs effectively.

NVIDIA continues to show strong operating efficiency, with revenue growing much faster than operating expenses. However, the gap has narrowed significantly, suggesting that its cost advantage is becoming less pronounced as the company grows.


## Cash Flow & Cash Conversion

### Business Question

Is reported performance supported by cash from operations?

### Approach

View: `performance`

- Operating_Cash_Flow_Margin_pct = OCF / Revenue
- Operating_Cash_Flow_Growth vs prior year
- Cash_Conversion_Ratio = OCF / Net Income (null if net income is 0)
- Free_Cash_Flow = OCF − Capital Expenditures
- Free_Cash_Flow_Margin_pct = FCF / Revenue

If net income is negative, treat cash conversion carefully. Do not read it as a normal conversion rate.


In [0]:
%sql
SELECT Company, Fiscal_Year, Net_Income, Cash_Flow_from_Operating_Activities, Operating_Cash_Flow_Margin_pct, Operating_Cash_Flow_Growth, Cash_Conversion_Ratio, Capital_Expenditures, Free_Cash_Flow, Free_Cash_Flow_Margin_pct
FROM financial_analytics.analytics.performance
where Fiscal_Year >= 2024
ORDER BY Company, Fiscal_Year;


Company,Fiscal_Year,Net_Income,Cash_Flow_from_Operating_Activities,Operating_Cash_Flow_Margin_pct,Operating_Cash_Flow_Growth,Cash_Conversion_Ratio,Capital_Expenditures,Free_Cash_Flow,Free_Cash_Flow_Margin_pct
AMD,2024,1.641E9,3.041E9,0.118,0.824,1.853,6.36E8,2.405E9,0.093
AMD,2025,4.335E9,7.709E9,0.223,1.535,1.778,9.74E8,6.735E9,0.194
Adobe,2024,5.56E9,8.056E9,0.375,0.103,1.449,1.83E8,7.873E9,0.366
Adobe,2025,7.13E9,1.0031E10,0.422,0.245,1.407,1.79E8,9.852E9,0.414
Apple,2024,9.3736E10,1.18254E11,0.302,0.07,1.262,9.447E9,1.08807E11,0.278
Apple,2025,1.1201E11,1.11482E11,0.268,-0.057,0.995,1.2715E10,9.8767E10,0.237
CrowdStrike,2024,7.2181E7,1.166207E9,0.382,0.239,16.157,1.76529E8,9.89678E8,0.324
CrowdStrike,2025,-1.5241E7,1.381727E9,0.349,0.185,-90.659,2.54852E8,1.126875E9,0.285
CrowdStrike,2026,-1.62502E8,1.612349E9,0.335,0.167,-9.922,3.02108E8,1.310241E9,0.272
Microsoft,2024,8.8136E10,1.18548E11,0.484,0.354,1.345,4.4477E10,7.4071E10,0.302


## Interpretation

AMD’s operating cash flow has strengthened significantly after declining in 2023. Its OCF margin increased to 22.3% in 2025, while operating cash flow grew 153.5%, indicating a strong improvement in cash generation.

Adobe has maintained strong operating cash generation, with its OCF margin recovering to 42.2% in 2025. Operating cash flow also grew 24.5%, indicating improving cash generation compared with the prior year.

Apple continues to generate strong operating cash flow, but its OCF margin declined to 26.8% in 2025 and operating cash flow fell 5.7%. This indicates some weakening in cash-generation efficiency compared with the prior year.

CrowdStrike continues to generate positive operating cash flow despite reporting negative net income. However, its OCF margin has declined to 33.5%, indicating that cash generation is becoming less efficient relative to revenue.

Microsoft has significantly strengthened its operating cash generation, with its OCF margin increasing to 55.1% in 2026. Operating cash flow also grew 34.4%, indicating strong conversion of revenue into operating cash.

NVIDIA has experienced exceptional growth in operating cash flow, with OCF increasing 60.3% in 2026. Although its OCF margin declined slightly to 47.6%, cash generation remains extremely strong relative to revenue.


## Financial Health & Risk

### Business Question

Does the company have the financial flexibility to support operations and growth if conditions tighten?

### Approach

View: `financial_health`

- Current_ratio
- debt_to_equity_ratio
- Liabilities_to_Assets
- ROA and ROE


In [0]:
%sql
SELECT company,Fiscal_Year,Current_ratio,debt_to_equity_ratio,Liabilities_to_Assets,roa,roe
FROM financial_analytics.analytics.financial_health
where Fiscal_Year >=2024
ORDER BY company, Fiscal_Year;


company,Fiscal_Year,Current_ratio,debt_to_equity_ratio,Liabilities_to_Assets,roa,roe
AMD,2024,2.616,0.03,0.168,0.024,0.029
AMD,2025,2.85,0.051,0.181,0.056,0.069
Adobe,2024,1.068,0.399,0.533,0.184,0.394
Adobe,2025,0.996,0.534,0.606,0.242,0.613
Apple,2024,0.867,1.697,0.844,0.257,1.646
Apple,2025,0.893,1.23,0.795,0.312,1.519
CrowdStrike,2024,1.764,0.322,0.648,0.011,0.031
CrowdStrike,2025,1.766,0.227,0.619,-0.002,-0.005
CrowdStrike,2026,1.773,0.168,0.597,-0.015,-0.037
Microsoft,2024,1.275,0.167,0.476,0.172,0.328


## Interpretation

AMD: Current ratio improved from 2.62 to 2.85, while liabilities-to-assets increased slightly from 16.8% to 18.1%. This indicates strong liquidity with a fair increase in balance-sheet obligations.

Adobe: Current ratio declined slightly below 1.0 in 2025, while debt-to-equity and liabilities-to-assets increased. This indicates weaker short-term liquidity and increasing balance-sheet leverage that may warrant attention.

Apple: Current ratio remained below 1.0, but debt-to-equity declined substantially from 1.70 to 1.23 and liabilities-to-assets also improved. This suggests reduced leverage, although short-term liquidity remains relatively tight.

CrowdStrike Current ratio remained strong at around 1.8, while debt-to-equity and liabilities-to-assets continued to decline. However, ROA moved into negative territory, indicating that profitability relative to assets has weakened.

Microsoft maintained a current ratio above 1.0 while debt-to-equity and liabilities-to-assets declined. ROA remained strong, although it decreased slightly in 2025 before recovering in 2026. These trends indicate that Microsoft maintained strong financial health while reducing its relative reliance on debt and liabilities

NVIDIA: NVIDIA has very strong liquidity and declining leverage, with debt-to-equity falling to 0.054 in 2026. ROA remains exceptionally strong despite declining from 65.3% to 58.1%.

## Peer Benchmarking

In [0]:
Select * from financial_analytics.analytics.peer_benchmarking
order by Average_Rank

company,fiscal_year,Growth_Rate_Rank,Growth_Differential_Rank,Operating_Margin_Rank,Net_Income_Margin_Rank,Operating_Cash_Flow_Margin_Rank,Operating_Expense_Growth_Rank,Current_Ratio_Rank,Debt_to_Equity_Rank,Liabilities_to_Assets_Rank,ROA_Rank,ROE_Rank,Average_Rank
NVIDIA,2025,1,6,1,1,1,6,1,2,2,1,2,1.778
Microsoft,2025,4,4,2,2,2,2,4,3,3,4,4,3.222
AMD,2025,2,1,5,5,6,4,2,1,1,5,5,3.556
Adobe,2025,5,3,3,3,3,1,5,5,4,3,3,3.667
Apple,2025,6,2,4,4,5,3,6,6,6,2,1,4.000
CrowdStrike,2025,3,5,6,6,4,5,3,4,5,6,6,4.778


##Managment Attention 

AMD should continue monitoring whether it can maintain its strong growth and cash generation as the company scales. Operating expenses are growing, but revenue is currently growing faster, so maintaining this relationship will be important for continued operating efficiency.

Adobe should closely monitor its current ratio, which has fallen from 1.068 to 0.996, to ensure it maintains sufficient current assets to meet short-term obligations. Management should continue monitoring current assets and liabilities to prevent further deterioration in short-term liquidity.

Apple should monitor its current assets and short-term liabilities to ensure its current ratio does not continue to decline. Although liquidity has improved from 0.87 to 0.89, the current ratio remains below 1.0, providing less flexibility for meeting short-term obligations. Apple should also monitor operating expense growth, which outpaced revenue growth by 1.8 percentage points in 2025 and could create additional pressure if the trend continues.

CrowdStrike should closely monitor its operating expenses, which reached 80.8% of revenue in 2026 and grew faster than revenue. Management should investigate the factors contributing to this cost pressure, as continued increases in operating expenses could negatively affect profitability. Net income margin has also declined from 2.4% in 2024 to -0.4% in 2025 and -3.4% in 2026, indicating increasing pressure on profitability.

Microsoft should continue monitoring its financial health to ensure its favorable leverage and liquidity trends are maintained. Debt-to-equity declined from 0.167 in 2024 to 0.091 in 2026, while liabilities-to-assets declined from 0.476 to 0.417. Maintaining these improvements while keeping the current ratio above 1.0 will help preserve the company’s financial flexibility.

NVIDIA should continue monitoring the pace of revenue growth as its exceptional growth begins to normalize. Revenue growth declined from 114% to 65%, but the company continues to generate strong profitability and operating cash flow. Management should monitor whether NVIDIA can maintain its strong profitability and cash generation as growth moderates and the company continues to scale.






##Conclusion

Overall, the analysis shows that company performance varies significantly across revenue growth, profitability, operating efficiency, cash generation, and financial health. NVIDIA and Microsoft demonstrated the strongest overall performance, although NVIDIA's exceptionally high revenue growth has begun to normalize while Microsoft has maintained more consistent growth and strong financial health.

AMD showed improving performance, with stronger revenue growth, improved operating efficiency, and significant growth in operating cash flow. Adobe maintained strong profitability and cash generation, but its declining short-term liquidity and increasing leverage warrant continued monitoring. Apple continued to demonstrate strong profitability and cash generation, but its current ratio remained below 1.0 and operating expenses continued to grow faster than revenue. CrowdStrike maintained strong revenue growth and positive operating cash flow, but increasing operating expenses and declining net income margins indicate growing pressure on profitability.

The analysis demonstrates that revenue growth alone does not provide a complete view of business performance. Evaluating growth alongside profitability, operating efficiency, cash generation, and financial health provides a more complete picture of how a company is performing and where management attention may be needed.



